In [37]:
# importação das utilidades para manipulação de dados
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

In [38]:
# definição do dataset iniciais
x = [[1,2], [3,4], [5,6], [7,8]]
y = [[3], [7], [11], [15]]

In [39]:
# conversão das listas para tensores float
x = torch.tensor(x).float()
y = torch.tensor(y).float()

In [40]:
# por algum motivo deu problema ai tive q comentar
#device = 'cuda' if torch.cuda.is_available() else 'cpu'
# move os tensores x e y para a memoria do dispositivo
# x = X.to(device)
# y = Y.to(device)

In [41]:
# definição de uma classe
class MyDataset(Dataset):
    def __init__(self, x, y):
        # converte as entradas para tensores float
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)
    # retorna o tamanho total do dataset
    def __len__(self):
        return len(self.x)
    # permite acessar um item específico do dataset
    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]

In [42]:
# instancia o dataset
ds = MyDataset(x, y)

/tmp/ipykernel_26161/1096084287.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.x = torch.tensor(x).float().to(device)
/tmp/ipykernel_26161/1096084287.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y).float().to(device)


In [43]:
# cria o dataloader para organizar o dataset
dl = DataLoader(ds, batch_size=2, shuffle=True)

In [44]:
# loop para iterar sobre o dataloader
for x,y in dl:
    print(x, y)

tensor([[5., 6.],
        [1., 2.]]) tensor([[11.],
        [ 3.]])
tensor([[7., 8.],
        [3., 4.]]) tensor([[15.],
        [ 7.]])


In [45]:
# definição da arquitetura da rede neural com duas camadas
class MyNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(8, 1)

    # define o fluxo de dados através das camadas
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

In [46]:
# instancia o modelo e define a função de perda
model = MyNeuralNet()
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.001)

In [47]:
losses = []
for _ in range(50):
    for data in dl: # Itera sobre o DataLoader
        opt.zero_grad() # Reinicia o gradiente
        x1, y1 = data
        loss_value = loss_fn(model(x1), y1)
        loss_value.backward() # Calcula o gradiente

        opt.step() # Atualiza os pesos
        losses.append(loss_value.detach().numpy())# Salva a perda para plotar depois